# Introduction to `tidyverse` Round 2
An ["opinionated" collection of R packages](https://tidyverse.org) for data science, driven by a coherent underlying design philosophy.
These packages are meant to help you with two essential processes:
1. **Data clean-up and organization**: Structure should be intuitive, so that it's easy to model, manipulate, and think about the data
2. **Data plotting**: The grammar of graphics (Week 13)

In [ ]:
library(tidyverse)

## Data preparation

### Loading the data

In [ ]:
data <- read_csv("data/148338_220209_095045_M057814.csv", skip=2)

In [ ]:
education_level <- data %>% pull(response) %>% first()  

data <- data %>%
    # Keep only useful columns
    select(c(rowNo, type, stim1, stim2, stimPos, trialType, response, RT)) %>%
    
    # Keep only useful rows
    filter(type != "form") %>%
    
    # Add demographic and trial-number info, turn trial type to factor
    mutate(
        education_level = education_level, # Add info
        trial_number = row_number(),
        trialType = factor(trialType, levels = c("incongruent", "congruent"))
    ) %>%
    
    # Rename trialType to trial_type
    rename(trial_type = trialType)

head(data)

### Tidying the data

Our data is almost tidy, but we still need to do several things.

#### Exercise 1
There are two columns that are uninformative. Remove them.

In [ ]:
data <- data %>%
  # Keep only useful columns
  select(c(rowNo, type, stim1, stim2, stimPos, trialType, response, RT)) %>%
  

#### Exercise 2
The columns `stim1` and `stim2` refer to the stimulus presented on the left and, respectively, right. Rename the columns to make them more informative.

In [ ]:
data <- data %>%
    rename(
        stim_left  = stim1,
        stim_right = stim2
    )

head(data)


#### Exercise 3
Add a new column called `subject_id` and set this participant to 1.

In [ ]:
data <- data %>%
    mutate(subject_id = 1)

head(data)


#### Exercise 4
Add a new column called `correct_side`. This column should encode the side of the correct answer (equivalently, the side of the smaller image). Use `str_detect(string, pattern)` to compute where the smaller image was on the screen and use a conditional `mutate` to fill in the values of `correct_side`.

In [ ]:
library(stringr)

data <- data %>%
    mutate(
        correct_side = case_when(
            str_detect(stim_left,  "small")  ~ "left",
            str_detect(stim_right, "small")  ~ "right",
            TRUE                              ~ NA_character_
        )
    )

head(data)

#### Exercise 5
Add a new column called `correct_key`. This column should be equal to `f` if the smaller image was on the left, and to `j` if the smaller image was on the right.

In [ ]:
data <- data %>%
    mutate(
        correct_key = case_when(
            correct_side == "left"  ~ "f",
            correct_side == "right" ~ "j",
            TRUE                    ~ NA_character_
        )
    )

head(data)


#### Exercise 6
Add a new column called `correct`. This column should be equal to 1 if the participant provided a correct response, 0 otherwise.

In [ ]:
data <- data %>%
    mutate(
        correct = if_else(response == correct_key, 1, 0)
    )

head(data)


#### Exercise 7
The experiment had 240 trials, equally divided into two blocks. Add a new column called `trial_block` that encodes this.

In [ ]:
data <- data %>%
    mutate(
        trial_block = if_else(trial_number <= 120, 1, 2)
    )

head(data)


#### Exercise 8
Recode `trial_number` so that it codes the trial number within a block. Instead of going from 1 to 240, it should go from 1 to 120 twice.

In [ ]:
data <- data %>%
    mutate(
        trial_number = ((trial_number - 1) %% 120) + 1
    )

head(data)


#### Exercise 9
It is good practice to use a consistent style throughout your script. One such style is called **snake**, the standard in python, which uses only lowercase letters and underscores: `variable_name`. Another common style is **camel**, the standard in JavaScript, which uses capital letters to mark the beginning of a new word: `variableName`. Our tibble at this point uses both styles, so turn all variable names to snake case. You can use `colnames` to see the vector of column names.

In [ ]:
library(janitor)

data <- data %>%
    clean_names()

colnames(data)


#### Exercise 10
Using `select`, reorder the columns such that participant information comes first, followed by trial block and number, followed by trial info.

In [ ]:
data <- data %>%
    select(
        subject_id, education_level,
        trial_block, trial_number,
        trial_type, stim_left, stim_right, stim_pos,
        correct_side, correct_key, response, correct, rt,
        row_no, type
    )

colnames(data)


#### Exercise 11
Concatenate all the commands in Exercises 1–10 into a single cell and store the output in a variable called `tidy_data`.

In [ ]:
library(tidyverse)
library(stringr)
library(janitor)

tidy_data <- read_csv("data/148338_220209_095045_M057814.csv", skip = 2) %>%
    
    # Extract education level before filtering
    { 
        education_level <- .$response[1]
        .
    } %>%
    
    # Keep only useful columns
    select(rowNo, type, stim1, stim2, stimPos, trialType, response, RT) %>%
    
    # Keep only useful rows
    filter(type != "form") %>%
    
    # Add demographic and trial information
    mutate(
        subject_id      = 1,
        education_level = education_level,
        trial_number    = row_number(),
        trial_block     = if_else(trial_number <= 120, 1, 2),
        trial_number    = ((trial_number - 1) %% 120) + 1,
        trialType       = factor(trialType, levels = c("incongruent", "congruent"))
    ) %>%
    
    # Rename stimulus columns
    rename(
        stim_left  = stim1,
        stim_right = stim2,
        trial_type = trialType
    ) %>%
    
    # Compute correct side and key
    mutate(
        correct_side = case_when(
            str_detect(stim_left,  "small") ~ "left",
            str_detect(stim_right, "small") ~ "right",
            TRUE                            ~ NA_character_
        ),
        correct_key = case_when(
            correct_side == "left"  ~ "f",
            correct_side == "right" ~ "j",
            TRUE                    ~ NA_character_
        ),
        correct = if_else(response == correct_key, 1, 0)
    ) %>%
    
    # Enforce snake_case naming
    clean_names() %>%
    
    # Reorder columns
    select(
        subject_id, education_level,
        trial_block, trial_number,
        trial_type, stim_left, stim_right, stim_pos,
        correct_side, correct_key, response, correct, rt,
        row_no, type
    )

head(tidy_data)


### Summary statistics

In [ ]:
tidy_data %>%
    summarize(rt = mean(rt), accuracy = mean(correct), error = mean(1 - correct))

In [ ]:
tidy_data %>%
    mutate(avg_error = mean(1 - correct))

#### Exercise 12
Using the `.by` argument in the call to `summarize`, find out if our participant showed the predicted size Stroop in reaction times and error rates.

In [ ]:
tidy_data %>%
    summarize(
        mean_rt    = mean(rt, na.rm = TRUE),
        mean_error = mean(1 - correct, na.rm = TRUE),
        .by = trial_type
    )


#### Exercise 13
Using a similar logic, find out if our participant also showed a SNARC effect: Was the participant faster when the small image was on the left?

In [ ]:
tidy_data %>%
    summarize(
        mean_rt    = mean(rt, na.rm = TRUE),
        mean_error = mean(1 - correct, na.rm = TRUE),
        .by = correct_side
    )


#### Exercise 14
Find out if the SNARC effect depends on trial type.

In [ ]:
tidy_data %>%
    summarize(
        mean_rt    = mean(rt, na.rm = TRUE),
        mean_error = mean(1 - correct, na.rm = TRUE),
        .by = c(trial_type, correct_side)
    )


## From a single participant to a full dataset
We load all the 12 csv files in the 'data' folder, then apply the `read_csv` function to each of them using `map_dfr`. The `.id` argument creates a column that keeps the information from each file separate. This is equivalent to having a subject_id, if there is one .csv file per participant.

In [ ]:
# Fetch all the files in the 'data' folder that end in .csv
raw_data <- list.files(path = 'data', pattern = ".csv$", full.names = TRUE) %>% 

  # Map the read_csv function to all of them, skipping the first 2 rows and creating a new id column called 'id' so that each file gets its own id
  # col_types = cols() just makes explicit that you want tidyverse to do its best to guess the type of each column (string, numeric, etc.)
  map_dfr(read_csv, col_types = cols(), skip = 2, .id = 'id') 

#### Exercise 15
Tidy the dataset exactly as we did for subject 1, while keeping the education-level information for each subject. Store it as `tidy_data`.

Hint: **group** the tibble before calling `first(response)`), then follow the same steps as before to obtain a tidy dataset. Use `ungroup()` to return to the tibble to the ungrouped state. In fact, in most of the exercises that follow, you will need to use grouping wisely.

```R
full_data <- raw_data %>% 
    mutate(education_level = first(response), .by = id) %>%
    ...
```

In [ ]:
library(dplyr)
library(stringr)
library(tidyverse)
# Fetch all the files in the 'data' folder that end in .csv
raw_data <- list.files(path = 'data', pattern = ".csv$", full.names = TRUE) %>% 
  
  # Map the read_csv function to all of them, skipping the first 2 rows and creating a new id column called 'id' so that each file gets its own id
  # col_types = cols() just makes explicit that you want tidyverse to do its best to guess the type of each column (string, numeric, etc.)
  map_dfr(read_csv, col_types = cols(), skip = 2, .id = 'id') 

#  Exercise 15

tidy_data <- raw_data %>%
  mutate(education_level = first(response), .by = id) %>%
  select(c(id, rowNo, type, stim1, stim2, stimPos, trialType, response, RT, education_level)) %>%
  
  filter(type != "form") %>%

  group_by(id) %>% # Grouping to ensure row_number() is per subject
  mutate(
    trial_number = row_number(), # Trial number within the subject's experiment
    trialType = factor(trialType, levels = c("incongruent", "congruent"))
  ) %>%
  ungroup() %>% # Ungroup after calculating trial_number
  
  rename(
    subject_id = id,
    stimulus_left = stim1,
    stimulus_right = stim2,
    stimulus_position = stimPos,
    trial_type = trialType,
    reaction_time = RT
  ) %>%
  
  select(-rowNo, -type) %>%
  
  mutate(
    correct_side = case_when(
      str_detect(stimulus_left, "Small") ~ "left",
      str_detect(stimulus_right, "Small") ~ "right",
      TRUE ~ NA_character_
    )
  ) %>%
  
  mutate(
    correct_key = case_when(
      correct_side == "left" ~ "f",
      correct_side == "right" ~ "j",
      TRUE ~ NA_character_
    )
  ) %>%
  

  mutate(
    correct = if_else(response == correct_key, 1, 0)
  ) %>%
  

  mutate(
    trial_block = if_else(trial_number <= 120, 1, 2)
  ) %>%
  

  mutate(
    trial_number = if_else(trial_block == 1, trial_number, trial_number - 120)
  ) %>%
  

  select(
    subject_id,
    education_level,
    trial_block,
    trial_number,
    trial_type,
    stimulus_left,
    stimulus_right,
    stimulus_position,
    
    correct_side,
    correct_key,
    response,
    
    correct,
    reaction_time
  )

print(head(tidy_data))
print(tail(tidy_data))
print(colnames(tidy_data))


#### Exercise 16
Trials where the responses were too slow or too fast should be excluded from the analysis. (Why?)  
Exclude the trials where the response is below 200 ms or higher than 1,500 ms. How many trials were excluded?

In [ ]:

rt_min <- 200
rt_max <- 1500

excluded_trials_count <- tidy_data %>%
  filter(reaction_time < rt_min | reaction_time > rt_max) %>%
  nrow()

trimmed_data <- tidy_data %>%
  filter(reaction_time >= rt_min, reaction_time <= rt_max)

cat(paste("Total trials excluded (RT <", rt_min, "ms or RT >", rt_max, "ms):", excluded_trials_count))

cat(paste("\nNew dataset (trimmed_data) rows:", nrow(trimmed_data)))

head(trimmed_data)

#### Exercise 17 
Exclude participants who didn't achieve 93% overall accuracy.  
How many subjects were excluded?

In [ ]:

subject_accuracy <- tidy_data %>%
  # group to calculate accuracy per subject
  group_by(subject_id) %>%
  summarise(
    accuracy = mean(correct),
    .groups = 'drop' # ungroup
  )

included_subjects <- subject_accuracy %>%
  filter(accuracy >= 0.93) %>%
  pull(subject_id)

trimmed_data_by_accuracy <- tidy_data %>%
  filter(subject_id %in% included_subjects)

total_subjects <- n_distinct(tidy_data$subject_id)
excluded_subjects_count <- total_subjects - length(included_subjects)

cat(paste("Total subjects in the original dataset:", total_subjects))
cat(paste("\nNumber of subjects excluded (Accuracy < 93%):", excluded_subjects_count))

cat(paste("\nNumber of subjects retained:", length(included_subjects)))
head(trimmed_data_by_accuracy)


# 1 subject excluded

#### Exercise 18
Summarize the response-time and accuracy measures by trial type to check whether there's a Stroop effect.

In [ ]:

stroop_summary <- trimmed_data_by_accuracy %>%
  summarise(

    mean_reaction_time = mean(reaction_time[correct == 1]),
    
    error_rate = 1 - mean(correct),
    total_trials = n(),
    
    .by = trial_type # group to get conditional statistics
  ) %>%
  mutate(overall_mean_rt = mean(mean_reaction_time))

print(stroop_summary)

stroop_rt <- stroop_summary %>% select(trial_type, mean_reaction_time)
stroop_error <- stroop_summary %>% select(trial_type, error_rate)

# calculate RT difference
congruent_rt <- stroop_rt %>% filter(trial_type == "congruent") %>% pull(mean_reaction_time)
incongruent_rt <- stroop_rt %>% filter(trial_type == "incongruent") %>% pull(mean_reaction_time)
rt_stroop_effect <- incongruent_rt - congruent_rt

# calculate error rate difference
congruent_error <- stroop_error %>% filter(trial_type == "congruent") %>% pull(error_rate)
incongruent_error <- stroop_error %>% filter(trial_type == "incongruent") %>% pull(error_rate)
error_stroop_effect <- incongruent_error - congruent_error

cat(paste("\nRT Stroop Effect (Incongruent - Congruent):", round(rt_stroop_effect, 2), "ms"))
cat(paste("\nError Rate Stroop Effect (Incongruent - Congruent):", round(error_stroop_effect, 4)))



## Changing the format of the data: `pivot_wider`, `pivot_longer`

#### Exercise 19: `pivot_wider`
Compute the average Stroop effects for each participant. One column should be called `stroop_rt`, the other should be called `stroop_error`. Using `pull`, extract the reaction-time Stroop vector and plot its histogram.

In [ ]:

library(tidyr)
library(ggplot2)


rt_summary <- trimmed_data_by_accuracy %>%
  filter(correct == 1) %>% # filter only correct responses for RT
  group_by(subject_id, trial_type) %>%
  summarise(
    mean_rt = mean(reaction_time),
    .groups = 'drop'
  ) %>%

  pivot_wider(
    names_from = trial_type,
    values_from = mean_rt,
    names_prefix = "RT_"
  ) %>%

  mutate(
    stroop_rt = RT_incongruent - RT_congruent
  ) %>%
  select(subject_id, stroop_rt)


error_summary <- trimmed_data_by_accuracy %>%
  group_by(subject_id, trial_type) %>%
  summarise(
    error_rate = 1 - mean(correct),
    .groups = 'drop'
  ) %>%

  pivot_wider(
    names_from = trial_type,
    values_from = error_rate,
    names_prefix = "Error_"
  ) %>%

  mutate(
    stroop_error = Error_incongruent - Error_congruent
  ) %>%
  select(subject_id, stroop_error)


stroop_wide <- left_join(rt_summary, error_summary, by = "subject_id")




# filter out NA values for plotting
rt_stroop_df <- stroop_wide %>% 
  select(stroop_rt) %>%
  filter(!is.na(stroop_rt))

mean_stroop_rt <- mean(rt_stroop_df$stroop_rt, na.rm = TRUE)

# Plot the histogram
stroop_histogram <- ggplot(rt_stroop_df, aes(x = stroop_rt)) +
  geom_histogram(binwidth = 25, fill = "darkblue", color = "white") +
  geom_vline(xintercept = mean_stroop_rt, 
             color = "red", 
             linetype = "dashed", 
             linewidth = 1) +
  labs(
    title = "Distribution of Individual Reaction Time Stroop Effects (Valid Data)",
    subtitle = paste0("Mean Stroop Effect: ", round(mean_stroop_rt, 2), " ms"),
    x = "Stroop Effect (Incongruent RT - Congruent RT in ms)",
    y = "Number of Participants"
  ) +
  theme_minimal()

print(stroop_histogram)


#### Exercise 20: `pivot_longer`
Building on the output tibble in Exercise 19, remove all columns except `id`, `stroop_rt`, and `stroop_error`, then gather the two stroop columns into a single column called `measure`. This column should take one of two values for each subject (`stroop_rt` or `stroop_error`), while the `value` column should register the respective participants' stroop effect.

In [ ]:

stroop_long <- stroop_wide %>%
  select(subject_id, stroop_rt, stroop_error) %>%
  
  # reshape the data
  pivot_longer(
    cols = c(stroop_rt, stroop_error), # The columns to gather/stack
    names_to = "measure",              # contains "stroop_rt" or "stroop_error"
    values_to = "value"                # contains the numerical effect size
  )

print(head(stroop_long, 4))

## Data plotting

### Rule 1: If your data is in the tidy format (one variable per column, one observation per row), plotting with `ggplot` will be very easy.

### Rule 2: No barplots.

### One possible way to do it

In [ ]:
average_data <- full_data %>% summarize(rt = mean(rt), .by = c(id, trial_type)) 

ggplot(average_data, aes(x = trial_type, y = rt, fill = trial_type)) +
  geom_boxplot(width = 0.5, alpha = 0.45) +
  geom_point(size = 2) +
  geom_line(aes(group = id), color = 'gray') +         
  stat_summary(fun.data = mean_se, linewidth = 2, shape = 21, size = 1.5) +
  labs(title = "Average reaction times by trial type (ms)", x = "Trial type", y = "") +
  theme_minimal() + 
  theme(
    legend.position = "none", 
    plot.title = element_text(face = "bold", size = 20),
    axis.title = element_text(size = 18),
    axis.text = element_text(size = 16))